# Google API’leri

[Google APIs paketi](https://pub.dev/packages/googleapis), Dart
projelerinden kullanabileceğiniz düzinelerce Google hizmetini sunar.

Bu sayfa, Google kimlik doğrulamasını kullanarak son kullanıcı
verileriyle etkileşime giren API’lerin nasıl kullanılacağını açıklar.

Kullanıcı verisi API’lerine örnek olarak Takvim, Gmail, YouTube ve
Firebase verilebilir.

> **Not** Flutter projenizden doğrudan kullanmanız gereken tek API’ler,
> Google kimlik doğrulamasını kullanarak kullanıcı verilerine
> erişenlerdir.
>
> [Servis hesapları](https://cloud.google.com/iam/docs/service-accounts)
> (service accounts) gerektiren API’ler, doğrudan bir Flutter
> uygulamasından **kullanılmamalıdır**. Bunu yapmak, servis kimlik
> bilgilerini uygulamanızın bir parçası olarak dağıtmayı gerektirir ki
> bu güvenli değildir. Bu API’leri kullanmak için bir ara servis
> oluşturmanızı öneririz.
>
> Firebase’e açıkça kimlik doğrulama eklemek için, [FirebaseUI
> kullanarak bir Flutter uygulamasına kullanıcı kimlik doğrulama akışı
> ekleme](https://firebase.google.com/codelabs/firebase-auth-in-flutter-apps)
> codelab’ine ve [Flutter’da Firebase Authentication’a
> Başlarken](https://firebase.google.com/docs/auth/flutter/start)
> dokümanlarına göz atın.

## Genel Bakış

Google API’lerini kullanmak için şu adımları izleyin:

1.  İstenen API’yi seçin.
2.  API’yi etkinleştirin.
3.  Kimlik doğrulaması yapın ve mevcut kullanıcıyı belirleyin.
4.  Kimliği doğrulanmış bir HTTP istemcisi (client) edinin.
5.  İstenen API sınıfını oluşturun ve kullanın.

## 1. İstenen API’yi seçin

[`package:googleapis`](https://pub.dev/packages/googleapis)
dokümantasyonu, her API’yi `isim_sürüm` formatında ayrı bir Dart
kütüphanesi olarak listeler. Örnek olarak
[`youtube_v3`](https://pub.dev/documentation/googleapis/latest/youtube_v3/youtube_v3-library.html)’e
göz atın.

Her kütüphane birçok tür sağlayabilir, ancak `Api` ile biten bir **kök**
sınıf vardır. YouTube için bu `YouTubeApi`dir.

Örneklendirmeniz (instantiate) gereken sınıf sadece `Api` sınıfı
değildir (bkz. adım 3), aynı zamanda API’yi kullanmak için gereken
izinleri temsil eden kapsamları (scopes) da sunar. Örneğin, `YouTubeApi`
sınıfının [Constants
bölümü](https://pub.dev/documentation/googleapis/latest/youtube_v3/YouTubeApi-class.html#constants),
mevcut kapsamları listeler. Bir son kullanıcının YouTube verilerini
okumak (ama yazmamak) için erişim talep etmek istiyorsanız, kullanıcıyı
`youtubeReadonlyScope` ile doğrulayın.

``` dart
/// `YouTubeApi` sınıfını sağlar.
import 'package:googleapis/youtube/v3.dart';
```

## 2. API’yi etkinleştirin

Google API’lerini kullanmak için bir Google hesabına ve bir Google
projesine sahip olmalısınız. Ayrıca istediğiniz API’yi etkinleştirmeniz
gerekir.

Bu örnek YouTube Data API v3’ü etkinleştirir. Ayrıntılar için [başlangıç
talimatlarına](https://developers.google.com/youtube/v3/getting-started)
bakın.

## 3. Kimlik doğrulaması yapın ve mevcut kullanıcıyı belirleyin

Kullanıcıları Google kimlikleriyle doğrulamak için
[`google_sign_in`](https://pub.dev/packages/google_sign_in) paketini
kullanın. Desteklemek istediğiniz her platform için oturum açma
yapılandırmasını yapın.

``` dart
/// `GoogleSignIn` sınıfını sağlar.
import 'package:google_sign_in/google_sign_in.dart';
```

Paketin işlevselliğine `GoogleSignIn` sınıfının statik bir örneği
üzerinden erişilir. Örnekle (instance) etkileşime geçmeden önce,
`initialize` metodu çağrılmalı ve tamamlanmasına izin verilmelidir.

``` dart
final _googleSignIn = GoogleSignIn.instance;

@override
void initState() {
  super.initState();
  _googleSignIn.initialize();
  // ···
}
```

Başlatma tamamlandıktan sonra ancak kullanıcı kimlik doğrulamasından
önce, bir kullanıcının oturum açıp açmadığını belirlemek için kimlik
doğrulama olaylarını dinleyin.

``` dart
GoogleSignInAccount? _currentUser;

@override
void initState() {
  super.initState();
  _googleSignIn.initialize().then((_) {
    _googleSignIn.authenticationEvents.listen((event) {
      setState(() {
        _currentUser = switch (event) {
          GoogleSignInAuthenticationEventSignIn() => event.user,
          _ => null,
        };
      });
    });
  });
}
```

İlgili kimlik doğrulama olaylarını dinlemeye başladıktan sonra, daha
önce oturum açmış bir kullanıcıyı doğrulamayı deneyebilirsiniz.

``` dart
void initState() {
  super.initState();
  _googleSignIn.initialize().then((_) {
    // ...
    // Daha önce oturum açmış bir kullanıcıyı doğrulamayı dene.
    _googleSignIn.attemptLightweightAuthentication();
  });
}
```

Yeni kullanıcıların da kimlik doğrulamasını sağlamak için
[`package:google_sign_in`](https://pub.dev/packages/google_sign_in)
tarafından sağlanan talimatları izleyin.

Bir kullanıcının kimliği doğrulandıktan sonra, kimliği doğrulanmış bir
HTTP istemcisi edinmelisiniz.

## 4. Kimliği doğrulanmış bir HTTP istemcisi edinin

Oturum açmış bir kullanıcınız olduğunda, uygulamanızın gerektirdiği API
kapsamları (scopes) için `authorizationForScopes` kullanarak ilgili
istemci yetkilendirme jetonlarını (tokens) isteyin.

``` dart
const relevantScopes = [YouTubeApi.youtubeReadonlyScope];
final authorization = await currentUser.authorizationClient
    .authorizationForScopes(relevantScopes);
```

> **Not** Eğer kapsamlarınız kullanıcı etkileşimi gerektiriyorsa,
> `authorizationForScopes` yerine bir etkileşim işleyicisinden
> `authorizeScopes` kullanmanız gerekecektir.

İlgili yetkilendirme jetonlarına sahip olduğunuzda, ilgili kimlik
bilgileri uygulanmış, kimliği doğrulanmış bir HTTP istemcisi kurmak için
[`package:extension_google_sign_in_as_googleapis_auth`](https://pub.dev/packages/extension_google_sign_in_as_googleapis_auth)
paketindeki `authClient` uzantısını kullanın.

``` dart
import 'package:extension_google_sign_in_as_googleapis_auth/extension_google_sign_in_as_googleapis_auth.dart';
final authenticatedClient = authorization!.authClient(
  scopes: relevantScopes,
);
```

## 5. İstenen API sınıfını oluşturun ve kullanın

İstenen API türünü oluşturmak ve metotları çağırmak için API’yi
kullanın. Örneğin:

``` dart
final youTubeApi = YouTubeApi(authenticatedClient);

final favorites = await youTubeApi.playlistItems.list(
  ['snippet'],
  playlistId: 'LL', // Liked List (Beğenilenler Listesi)
);
```

## Daha fazla bilgi

Şunlara göz atmak isteyebilirsiniz:

- [`extension_google_sign_in_as_googleapis_auth`
  örneği](https://pub.dev/packages/extension_google_sign_in_as_googleapis_auth/example),
  bu sayfada açıklanan kavramların çalışan bir uygulamasıdır.

## 📄 Lisans Bilgisi

Bu doküman, **Flutter resmi dokümantasyonundan** türetilmiş Türkçe ders
notudur.

**Orijinal kaynak:**  
https://docs.flutter.dev/data-and-backend/google-apis

**Türkçe çeviri ve düzenleme:**  
[Doç. Dr. Hakan Temiz](mailto:htemiz@artvin.edu.tr)

------------------------------------------------------------------------

### Lisans Kapsamı

Bu dokümandaki içerikler aşağıdaki açık lisanslar kapsamında
sunulmaktadır:

**Metin içerikleri (anlatım ve açıklamalar):**  
Flutter resmi dokümantasyonundan alınmış veya uyarlanmıştır.  
**Lisans:** Creative Commons Attribution 4.0 International (CC BY 4.0)  
https://creativecommons.org/licenses/by/4.0/

Bu lisans kapsamında: - İçerik kopyalanabilir, dağıtılabilir ve
uyarlanabilir  
- Ticari kullanım serbesttir  
- Kaynak belirtilmesi zorunludur

**Kod örnekleri:**  
Flutter resmi dokümantasyonundan alınmış veya uyarlanmıştır.  
**Lisans:** BSD 3-Clause License  
https://opensource.org/licenses/BSD-3-Clause

Bu lisans kapsamında: - Kodlar kopyalanabilir, değiştirilebilir ve
dağıtılabilir  
- Ticari kullanım serbesttir  
- Lisans bildiriminin korunması gerekir